In [5]:
import sys
import json
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## src/config.py

In [3]:
%%writefile ../src/config.py
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
RAW_PATH = ROOT / "data" / "raw" / "students.csv"
PROCESSED_DIR = ROOT / "data" / "processed"
TRAIN_PATH = PROCESSED_DIR / "train.csv"
TEST_PATH = PROCESSED_DIR / "test.csv"
MODELS_DIR = ROOT / "models"
REPORTS_DIR = ROOT / "reports"

RANDOM_STATE = 42
TARGET = "Mental_Health_Risk"
LABELS = ["Low", "Medium", "High"]
LABEL_TO_INT = {label: i for i, label in enumerate(LABELS)}

NUMERIC_COLS = [
    "Age", "Year_of_Study", "Sleep_Hours", "Study_Hours_Per_Week", "GPA",
    "Social_Media_Hours", "Screen_Time_Hours", "Financial_Stress",
    "Family_Support", "Attendance_Rate",
]
ENGINEERED_COLS = ["Sleep_Deficit", "Non_Social_Screen_Time", "Stress_Support_Gap"]

ORDINAL_COLS = {
    "Physical_Activity": ["Low", "Moderate", "High"],
    "Diet_Quality": ["Poor", "Average", "Good"],
    "Part_Time_Job": ["No", "Yes"],
    "Previous_Mental_Health_History": ["No", "Yes"],
}
NOMINAL_COLS = ["Gender"]

INPUT_COLUMNS = NUMERIC_COLS + list(ORDINAL_COLS) + NOMINAL_COLS

VALID_RANGES = {
    "Age": (16, 40),
    "Year_of_Study": (1, 5),
    "Sleep_Hours": (2, 12),
    "Study_Hours_Per_Week": (0, 80),
    "GPA": (0, 4),
    "Social_Media_Hours": (0, 16),
    "Screen_Time_Hours": (0, 18),
    "Financial_Stress": (1, 10),
    "Family_Support": (1, 10),
    "Attendance_Rate": (0, 100),
}

Overwriting ../src/config.py


## src/utils.py

In [4]:
%%writefile ../src/utils.py
import pandas as pd
from src.config import TARGET, LABEL_TO_INT


def load_xy(path):
    df = pd.read_csv(path)
    X = df.drop(columns=[TARGET])
    y = df[TARGET].map(LABEL_TO_INT)
    return X, y

Overwriting ../src/utils.py


## src/preprocessing.py

In [7]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler)

from src.config import (NUMERIC_COLS, ENGINEERED_COLS, ORDINAL_COLS, NOMINAL_COLS)


def add_features(df):
    df = df.copy()
    df["Sleep_Deficit"] = (7 - df["Sleep_Hours"]).clip(lower=0)
    df["Non_Social_Screen_Time"] = (
        df["Screen_Time_Hours"] - df["Social_Media_Hours"]).clip(lower=0)
    df["Stress_Support_Gap"] = df["Financial_Stress"] - df["Family_Support"]
    return df


class IQRClipper(BaseEstimator, TransformerMixin):

    def __init__(self, factor=1.5):
        self.factor = factor

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        q1, q3 = np.percentile(X, [25, 75], axis=0)
        iqr = q3 - q1
        self.lower_ = q1 - self.factor * iqr
        self.upper_ = q3 + self.factor * iqr
        return self

    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.lower_, self.upper_)



def build_preprocessor():
    numeric = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clipper", IQRClipper(factor=1.5)),
        ("scaler", StandardScaler()),
    ])
    ordinal = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(
            categories=list(ORDINAL_COLS.values()),
            handle_unknown="use_encoded_value", unknown_value=-1,
        )),
    ])
    nominal = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer([
        ("num", numeric, NUMERIC_COLS + ENGINEERED_COLS),
        ("ord", ordinal, list(ORDINAL_COLS)),
        ("nom", nominal, NOMINAL_COLS),
    ])


def build_feature_prep():
    return Pipeline([
        ("features", FunctionTransformer(add_features)),
        ("preprocessor", build_preprocessor()),
    ])


def build_pipeline(model):
    return Pipeline([
        ("prep", build_feature_prep()),
        ("model", model),
    ])

## nettoyage et split 80/20

In [9]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src.config import *

def clean(df):
    df = df.drop(columns=["Student_ID"], errors="ignore")


    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]):
            df[col] = df[col].str.strip()

    df = df.drop_duplicates()
    df = df.dropna(subset=[TARGET])

    for col, (low, high) in VALID_RANGES.items():
        invalid = df[col].notna() & ~df[col].between(low, high)
        print(f"{col}: {invalid.sum()} valeur(s) hors bornes")
        df.loc[invalid, col] = np.nan
    return df

raw = pd.read_csv(RAW_PATH)
print("Avant nettoyage :", raw.shape)

df = clean(raw)
print("=============================================")
print("Après nettoyage :", df.shape)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
train, test = train_test_split(
    df, test_size=0.2, stratify=df[TARGET], random_state=RANDOM_STATE)

train.to_csv(TRAIN_PATH, index=False)
test.to_csv(TEST_PATH, index=False)

print("Train :", train.shape, "| Test :", test.shape)

Avant nettoyage : (2000, 17)
Age: 0 valeur(s) hors bornes
Year_of_Study: 0 valeur(s) hors bornes
Sleep_Hours: 0 valeur(s) hors bornes
Study_Hours_Per_Week: 0 valeur(s) hors bornes
GPA: 0 valeur(s) hors bornes
Social_Media_Hours: 0 valeur(s) hors bornes
Screen_Time_Hours: 0 valeur(s) hors bornes
Financial_Stress: 0 valeur(s) hors bornes
Family_Support: 0 valeur(s) hors bornes
Attendance_Rate: 0 valeur(s) hors bornes
Après nettoyage : (2000, 16)
Train : (1600, 16) | Test : (400, 16)


## vérification du split

In [10]:
print("Répartition des classes (train) :")
print(train[TARGET].value_counts(normalize=True).round(3))
print("\nRépartition des classes (test) :")
print(test[TARGET].value_counts(normalize=True).round(3))
print("\nManquants dans le train (%) :")
print((train.isna().mean() * 100).round(2).loc[lambda s: s > 0])

Répartition des classes (train) :
Mental_Health_Risk
High      0.34
Medium    0.33
Low       0.33
Name: proportion, dtype: float64

Répartition des classes (test) :
Mental_Health_Risk
High      0.34
Medium    0.33
Low       0.33
Name: proportion, dtype: float64

Manquants dans le train (%) :
Sleep_Hours           3.38
GPA                   2.25
Social_Media_Hours    3.00
Financial_Stress      4.75
Family_Support        2.25
Attendance_Rate       2.44
Diet_Quality          3.44
dtype: float64
